In [ ]:
#Referenz ⇒ Canonical Form 
#pdb ID in der pdb suchen ⇒ FASTA sequence file downloaden  
#bei SAbPred: SCALOP in Submission form die Datei hochladen 
#results: für jedes CDR (H1, H2, L1, L2, L3) erkennt er CDR Sequenz aus der gesamten Sequenz (mit Antibody und antigen) und gibt einem canonical form und median structure 
#(L1-11-A → Canonical form A für eine L1-Schleife mit 11 Aminosäuren)
#muss man für alle unsere pdb Einträge machen und dann canonical cluster nochmal im code definieren (also die pdb Einträge zuordnen)
#dann der vergleich mit V-measure

In [ ]:
!git clone https://github.com/oxpig/SCALOP.git
!cd SCALOP

import sys
sys.path.append('https://github.com/oxpig/SCALOP.git')
import SCALOP

import csv
#import scalop
from SCALOP.predict import assign

!conda install -c bioconda hmmer 

# Beispiel: Liste von Antikörpersequenzen (hier Dummy-Sequenzen)
sequences = [
    ("antibody1", "EVQLVESGGGLVQPGGSLRLSCAASGFTFSSYAMSWVRQAPGKGLEWVSAISSGSGGSTYYADSVKGRF"),
    ("antibody2", "QVQLVQSGAEVKKPGASVKVSCKASGYTFTSYWMHWVRQAPGQGLEWMGGIIPIFGTANYAQKFQGRVTMTRDTSISTAYLQWSSLKASDTAIYYCAGRGYYSWGQGTLVTVSS")
]

# Ergebnis-Datei
output_file = "canonical_forms_output.csv"

# Schreibe Ergebnisse in CSV mit folgenden Spalten:
# Antibody_ID, CDR_Name, CDR_Sequence, Canonical_Form
with open(output_file, mode="w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Antibody_ID", "CDR_Name", "CDR_Sequence", "Canonical_Form"])

    for ab_id, seq in sequences:
        result = assign(seq, scheme="chothia", definition="chothia")

        for cdr_name, cdr_info in result.items():
            positions = cdr_info['positions']
            if not positions:
                continue
            cdr_seq = ''.join([seq[i-1] for i in positions])  # Chothia-Nummerierung ist 1-basiert
            cluster = cdr_info['cluster']

            writer.writerow([ab_id, cdr_name, cdr_seq, cluster])

print(f"Alle CDRs und Canonical Forms wurden in {output_file} gespeichert.")




In [ ]:
!git clone https://github.com/oxpig/SCALOP.git
!conda create -n scalop-env python=3.8 -y
!conda activate scalop-env
%conda install -c bioconda numpy pandas hmmer biopython -y
%pip install SCALOP/

In [ ]:
!conda config --add channels conda-forge
!conda config --add channels bioconda
!conda config --set channel_priority strict
!conda update -n base -c defaults conda
%conda install -c bioconda hmmer 


In [ ]:
from scalop.predict import assign
input='VKLLEQSGAEVKKPGASVKVSCKASGYSFTSYGLHWVRQAPGQRLEWMGWISAGTGNTKYSQKFRGRVTFTRDTSATTAYMGLSSLRPEDTAVYYCARDPYGGGKSEFDYWGQGTLVTVSS/ELVMTQSPSSLSASVGDRVNIACRASQGISSALAWYQQKPGKAPRLLIYDASNLESGVPSRFSGSGSGTDFTLTISSLQPEDFAIYYCQQFNSYPLTFGGGTKVEIKRTV'
assign(input)

In [ ]:
#fasta dateien downloaden

import requests      #Modul zum Herunterladen von Daten aus dem Internet
import os            #Modul für Dateipfade und Ordnerverwaltung

def download_fasta(pdb_id, outdir="fasta_files"):
    """
    Lädt die FASTA-Sequenzdatei für einen gegebenen PDB-Eintrag
    von der RCSB PDB-Website herunter und speichert sie lokal.

    Parameter:
    - pdb_id: z.B. "1abc" (Groß-/Kleinschreibung egal)
    - outdir: Zielordner, in dem die FASTA-Dateien gespeichert werden

    Rückgabe:
    - Pfad zur gespeicherten FASTA-Datei (oder None bei Fehler)
    """

    #URL zur FASTA-Datei auf der rcsb.org-Website (liefert alle Chains)
    url = f"https://www.rcsb.org/fasta/entry/{pdb_id}/display"

    #HTTP-GET-Request an die URL schicken
    response = requests.get(url)

    #Prüfen ob der Download erfolgreich war (Statuscode 200 = OK)
    if response.status_code == 200:

        #Zielordner anlegen, falls er noch nicht existiert
        os.makedirs(outdir, exist_ok=True)

        #Speicherpfad für die Datei zusammensetzen
        fasta_path = os.path.join(outdir, f"{pdb_id}.fasta")

        #Inhalt in Datei schreiben
        with open(fasta_path, "w") as f:
            f.write(response.text)

        #Pfad zur fertigen Datei zurückgeben
        return fasta_path
        

    else:
        #Fehlerausgabe, falls Download fehlgeschlagen
        print(f"Fehler beim Herunterladen von {pdb_id} (Status: {response.status_code})")
        return None
    
    
    

In [ ]:
#hochladen auf sabpred und extraktion der canonical forms

In [ ]:
import time
import pandas as pd
%pip install selenium webdriver-manager

# Automatisierung mit Selenium. Anforderung: selenium und webdriver-manager installiert
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Hilft, automatisch passenden ChromeDriver zu finden
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [ ]:
from selenium.webdriver.common.action_chains import ActionChains
import matplotlib.pyplot as plt
#pdb ids aus unserer geflterten datei laden
df = pd.read_csv("data/ab_ag_annotated.tsv", sep="\t").drop_duplicates(subset = ["pdb", "CDR_H1", "CDR_H2", "CDR_L1", "CDR_L2", "CDR_L3"], ignore_index = True)
pdb_ids = df["pdb"].unique() #nur eindeutige PDB-IDs extrahieren

#automatisierten brwoser starten
# Chrome-Optionen: "headless" = läuft ohne sichtbares Fenster
options = webdriver.ChromeOptions()
#options.add_argument('--headless')  # unsichtbar im Hintergrund => hab es sichtbar aus probiert => der link zur startseite öffnet sich auf jeden fall

# Browser starten mit automatisch installiertem Treiber (=Schnittstelle zwischen python skript und browser)
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

results = []  # Hier wird jedes Canonical-Form-Ergebnis als dict gespeichert

#Iteration über PDB-IDs
for pdb_id in pdb_ids:
    download_fasta(pdb_id)

    # Gehe zur sappred-Webseite
    driver.get("https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabpred")
    

   #warten damit auf seite auch alles geladen ist
    time.sleep(5)

    # Suche Link zu SCALOP über XPATH und überprüfe, ob Element vorhanden ist
    try:
        scalop_link = WebDriverWait(driver, 20).until(
            EC.element_to_be_clickable((By.LINK_TEXT, "SCALOP"))
        )
        driver.execute_script("arguments[0].scrollIntoView();", scalop_link)
        scalop_link.click()
        print("SCALOP Link erfolgreich geklickt!")
    except Exception as e:
        print("Konnte SCALOP Link nicht klicken:", e)

    #überprüfung wo auf seite wir uns befinden
    #driver.save_screenshot('debug.png')
    #img = plt.imread('debug.png')
    #plt.figure(figsize=(10, 8))
    #plt.imshow(img)
    #plt.axis('off')
    #plt.show()
    
    #navigation auf der scalop seite
    # Warte bis Formular geladen ist
    WebDriverWait(driver, 20).until(
    EC.presence_of_element_located((By.XPATH, '//input[@type="file"]'))
    )

    # Scrolle nach unten
    actions = ActionChains(driver)
    actions.move_to_element(driver.find_element(By.XPATH, '//input[@type="file"]')).perform()

    # Lade FASTA-Datei hoch aber mit wartezeit damit die seite auch wirklich geladen ist
    file_input = WebDriverWait(driver, 10).until(
    EC.visibility_of_element_located((By.ID, "fastafile"))
    )

    file_path=os.path.abspath('fasta_files\\9ds2.fasta') #os.path.abspath gibt den absoluten Pfad zur Datei zurück
    file_input.send_keys(file_path) #schickt den Pfad der Datei an dieses Feld
    
    #überprüfung ob die datei hochgeladen wurde und wo wir uns befinden
    #driver.save_screenshot('debug.png')
    #img = plt.imread('debug.png')
    #plt.figure(figsize=(10, 8))
    #plt.imshow(img)
    #plt.axis('off')
    #plt.show()

    # Wähle Chothia bei Numbering Scheme
    # Scrolle zu dem Element
    chothia_radio = driver.find_element(By.XPATH, '//input[@type="radio" and @value="chothia"]')
    driver.execute_script("arguments[0].scrollIntoView();", chothia_radio)

    # Nun anklicken
    chothia_radio.click()

    #überprüfung ob chothia ausgewählt wurde und wo wir uns befinden
    driver.save_screenshot('debug.png')
    img = plt.imread('debug.png')
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    # Wähle Chothia bei CDR Definition
    #cdr_chothia_radio = driver.find_element(By.XPATH, '//input[@type="radio" and @value="chothia"]')

    cdr_chothia_radio = driver.find_element(By.XPATH, '//input[@name="cdrdef" and @value="chothia"]')
    cdr_chothia_radio.click()


    driver.save_screenshot('debug.png')
    img = plt.imread('debug.png')
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.show()

    # Klicke Assign-Button
    assign_button = driver.find_element(By.XPATH, '//button[contains(text(), "Assign")]')
    assign_button.click()

    print(f"Starte Verarbeitung für {pdb_id}")
    # Ergebnisse speichern
    data = []

    # Warte bis die Ergebnisse sichtbar sind
    WebDriverWait(driver, 20).until(
        EC.presence_of_all_elements_located((By.XPATH, '//div[contains(@class, "accordion-heading")]'))
    )

    # Finde alle blauen Kästen
    blue_boxes = driver.find_elements(By.XPATH, '//div[contains(@class, "accordion-heading")]')

    for box in blue_boxes:
        try:
            # Scrolle zum Kasten und klicke ihn
            driver.execute_script("arguments[0].scrollIntoView();", box)
            time.sleep(1)
            box.click()
            time.sleep(1)

            # Warte bis Tabelle sichtbar ist
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.XPATH, './/table[contains(@class, "table-results")]'))
            )

            # Finde alle Zeilen der Tabelle innerhalb dieses Kastens
            rows = box.find_elements(By.XPATH, './/table[contains(@class, "table-results")]/tbody/tr')
            
            for row in rows:
                cols = row.find_elements(By.TAG_NAME, 'td')
                if len(cols) >= 4:
                    cdr = cols[0].text.strip()
                    cdr_seq = cols[1].text.strip()
                    canonical_form = cols[2].text.strip()
                    median_struct = cols[3].text.strip()
                    
                    data.append({
                        'CDR': cdr,
                        'CDR_sequence': cdr_seq,
                        'Canonical_Form': canonical_form,
                        'Median_Structure': median_struct
                    })
        except Exception as e:
            print("Fehler beim Verarbeiten eines Kastens:", e)
            continue

    # Ergebnisse in DataFrame speichern
    df = pd.DataFrame(data)
    df.to_csv("scalop_results.csv", index=False)
    print(df)

        

#browser schließen und ergbnisse speichern
# Browser schließen
driver.quit()

# Ergebnisse in DataFrame umwandeln und abspeichern
df_results = pd.DataFrame(results)
df_results.to_csv("scalop_canonical_forms.csv", index=False)


In [ ]:
#überprüfen ob scalop zu finden ist in diesem link 
#wartezeit damit dynamische inhalte zeit haben geladen zu werden
import time

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)
# Gehe zur Hauptseite
driver.get("https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabpred")

# Warte 5 Sekunden für vollständiges Laden der Seite
time.sleep(5)

# HTML-Code ausgeben, um zu prüfen, ob der Link im DOM ist
#html = driver.page_source
#print(html)

# Suche Link zu SCALOP über XPATH und überprüfe, ob Element vorhanden ist
try:
    scalop_link = WebDriverWait(driver, 20).until(
        EC.element_to_be_clickable((By.LINK_TEXT, "SCALOP"))
    )
    driver.execute_script("arguments[0].scrollIntoView();", scalop_link)
    scalop_link.click()
    print("SCALOP Link erfolgreich geklickt!")
except Exception as e:
    print("Konnte SCALOP Link nicht klicken:", e)

#nach elementen mit file input suchen um fasta datei hochladen zu können
elements = driver.find_elements(By.XPATH, '//input[@type="file"]')
print(f"Gefundene Elemente: {len(elements)}")
for e in elements:
    print(e.get_attribute('outerHTML'))

    #es gibt also 2 elemnte wir müssen das richtige ansteuern 

#nach elementen suchen um auf chotia zu klicken


In [ ]:
#fasta downladen
download_fasta("9ds2")

In [ ]:
os.path.exists('fasta_files\\9ds2.fasta')